# ✍️ W3: F&B 메뉴 카피라이팅 자동화
**hexa-2 — Week 3** | ⏱️ 60분 | 🎯 배달앱 설명 + SNS 캡션 + ZIP

## Step 0: 환경 설정

In [ ]:
import subprocess
subprocess.run(['apt-get','-qq','-y','install','fonts-nanum'],capture_output=True)
!pip install -q google-generativeai pandas matplotlib
import matplotlib.pyplot as plt, matplotlib.font_manager as fm, pandas as pd
_n=[f for f in fm.findSystemFonts() if 'Nanum' in f]
if _n: fm.fontManager.addfont(_n[0]); plt.rcParams['font.family']=fm.FontProperties(fname=_n[0]).get_name()
plt.rcParams['axes.unicode_minus']=False
try:
    from google.colab import userdata; API_KEY=userdata.get('GEMINI_API_KEY')
except: API_KEY=input('GEMINI_API_KEY: ')
import google.generativeai as genai; genai.configure(api_key=API_KEY)
model=genai.GenerativeModel('gemini-2.0-flash'); print('✅ 준비 완료')


## Step 1: 가게 & 메뉴 정보 입력 (✏️)

In [ ]:
STORE={'name':'✏️ [가게명]','style':'✏️ [한식|중식|일식|양식|분식]'}
MENUS=[
    {'name':'✏️ [메뉴1]','price':12000,'features':'✏️ [재료/특징]'},
    {'name':'✏️ [메뉴2]','price':15000,'features':'✏️ [재료/특징]'},
    {'name':'✏️ [메뉴3]','price':9000,'features':'✏️ [재료/특징]'},
]
print(f'✅ {STORE["name"]} — {len(MENUS)}개 메뉴')


## Step 2: 배달앱 메뉴 설명 자동 생성 (80자 이내)

In [ ]:
results=[]
for menu in MENUS:
    p=f"""배달앱 메뉴 설명. 80자 이내, 식욕 자극, 핵심 재료 포함.
가게:{STORE['name']}({STORE['style']}), 메뉴:{menu['name']}({menu['price']:,}원), 특징:{menu['features']}"""
    desc=model.generate_content(p).text.strip()[:80]
    p2=f"""인스타그램 게시글. 이모지3개+해시태그5개 포함. 150자 이내.
메뉴:{menu['name']}, 가게:{STORE['name']}"""
    instagram=model.generate_content(p2).text.strip()
    results.append({'메뉴':menu['name'],'가격':menu['price'],'배달앱설명':desc,'인스타':instagram})
    print(f"[{menu['name']}]\n배달앱: {desc}\n인스타: {instagram[:80]}...\n")


## Step 3: 결과 DataFrame 표시

In [ ]:
import pandas as pd
result_df=pd.DataFrame(results)
print(result_df[['메뉴','가격','배달앱설명']].to_string(index=False))


## Step 4: ZIP 다운로드

In [ ]:
import zipfile
for r in results:
    with open(f'{r["메뉴"]}_copy.txt','w',encoding='utf-8') as f:
        f.write(f'=== {r["메뉴"]} ({r["가격"]:,}원) ===\n\n')
        f.write(f'[배달앱 설명]\n{r["배달앱설명"]}\n\n[인스타그램]\n{r["인스타"]}')
with zipfile.ZipFile('menu_copy.zip','w') as z:
    for r in results: z.write(f'{r["메뉴"]}_copy.txt')
result_df.to_csv('menu_copy.csv',index=False,encoding='utf-8-sig')
from google.colab import files
files.download('menu_copy.zip'); files.download('menu_copy.csv')
print('✅ 완료!')
